# LangGraph 기초 + MLflow 트레이싱
상태(State) · 노드(Node) · 엣지(Edge) 로 워크플로를 만들고, 마지막에 `create_agent` 로 도구 에이전트를 만든다.

# **1. 환경 설정**

In [ ]:
from common import setup_mlflow, get_model

setup_mlflow()
model = get_model()

# **2. StateGraph 기초 (LLM 없이)**
노드는 `state -> 부분 업데이트(dict)` 함수. 리듀서(`operator.add`)를 붙인 필드는 덮어쓰지 않고 누적된다.  
API 키 없이도 돌아가므로 그래프 구조부터 익히기 좋다.

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class CountState(TypedDict):
    n: int
    log: Annotated[list[str], operator.add]   # 누적

def inc(state: CountState):
    return {"n": state["n"] + 1, "log": [f"inc -> {state['n'] + 1}"]}

def should_continue(state: CountState):
    return "inc" if state["n"] < 3 else END

builder = StateGraph(CountState)
builder.add_node("inc", inc)
builder.add_edge(START, "inc")
builder.add_conditional_edges("inc", should_continue, ["inc", END])
counter = builder.compile()

counter.invoke({"n": 0, "log": []})

In [ ]:
# 그래프 구조 확인 (Mermaid)
print(counter.get_graph().draw_mermaid())

# **3. LLM 노드 + 조건 분기 (라우팅)**
질문을 분류한 뒤 분류 결과에 따라 다른 노드로 보낸다.

In [ ]:
from typing import Literal
from pydantic import BaseModel

class Route(BaseModel):
    kind: Literal["code", "concept"]

class QAState(TypedDict):
    question: str
    kind: str
    answer: str

router = model.with_structured_output(Route)

def classify(state: QAState):
    r = router.invoke(f"다음 질문이 코드 작성 요청이면 code, 개념 설명이면 concept: {state['question']}")
    return {"kind": r.kind}

def answer_code(state: QAState):
    return {"answer": model.invoke(f"파이썬 코드로만 답해: {state['question']}").text}

def answer_concept(state: QAState):
    return {"answer": model.invoke(f"비유를 들어 세 줄로 설명해: {state['question']}").text}

b = StateGraph(QAState)
b.add_node("classify", classify)
b.add_node("code", answer_code)
b.add_node("concept", answer_concept)
b.add_edge(START, "classify")
b.add_conditional_edges("classify", lambda s: s["kind"], ["code", "concept"])
b.add_edge("code", END)
b.add_edge("concept", END)
qa = b.compile()

out = qa.invoke({"question": "리스트를 역순으로 정렬하는 법"})
print(out["kind"]); print(out["answer"])

# **4. create_agent: 도구 에이전트 + 메모리**
`create_agent` 는 LangGraph 위에서 "모델 → 도구 실행 → 모델 …" 루프를 만들어 준다.  
`checkpointer` 를 주면 `thread_id` 별로 대화가 이어진다.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def multiply(a: float, b: float) -> float:
    """두 수를 곱한다."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 알려준다."""
    return f"{city}: 맑음, 21도"  # 실습용 가짜 데이터

agent = create_agent(
    model,
    tools=[multiply, get_weather],
    system_prompt="너는 한국어로 답하는 조수야. 필요하면 도구를 써.",
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "study-1"}}

r = agent.invoke({"messages": [{"role": "user", "content": "부산 날씨 어때? 그리고 12.5 곱하기 8 은?"}]}, config)
print(r["messages"][-1].text)

In [ ]:
# 같은 thread_id 로 이어서 질문 → 이전 대화를 기억
r = agent.invoke({"messages": [{"role": "user", "content": "방금 계산한 값에 2를 곱하면?"}]}, config)
print(r["messages"][-1].text)

# **5. MLflow 에서 트레이스 확인**
에이전트 트레이스를 열면 모델 호출 → 도구 실행 → 모델 호출이 트리로 보인다.

In [ ]:
import mlflow

mlflow.search_traces(max_results=10)[["trace_id", "request_time", "state", "execution_duration"]]

# **99. 참고자료**
- LangGraph 문서: https://docs.langchain.com/oss/python/langgraph/overview
- create_agent: https://docs.langchain.com/oss/python/langchain/agents
- MLflow LangGraph 트레이싱: https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langgraph